# Overfitting With Polynomial Features

This notebook compares polynomial models with different degrees. The models all use the same `LinearRegressionGD`; only the feature expansion changes.

Author: Pasquale Marzaioli

## Key Idea

A low-degree polynomial can underfit by being too simple. A very high-degree polynomial can overfit by following noise. Test error helps reveal the difference.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from ml_from_scratch import LinearRegressionGD, mean_squared_error
from ml_from_scratch.datasets import make_polynomial_regression_data
from ml_from_scratch.preprocessing import (
    normalize_features,
    polynomial_features,
    train_test_split,
    transform_features,
)

## Generate Curved Data

The true pattern is quadratic, with noise added to make the task realistic.

In [ ]:
X, y = make_polynomial_regression_data(
    n_samples=80,
    degree=2,
    coefficients=np.array([1.0, -2.0, 0.5]),
    noise=1.0,
    random_state=11,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=11,
)

print(X_train.shape, X_test.shape)

## Train Several Polynomial Degrees

Each degree expands the same raw `x` values into powers, normalizes those columns, then trains linear regression.

In [ ]:
def train_degree(degree):
    X_train_poly = polynomial_features(X_train, degree=degree)
    X_test_poly = polynomial_features(X_test, degree=degree)
    X_train_norm, means, scales = normalize_features(X_train_poly)
    X_test_norm = transform_features(X_test_poly, means, scales)

    model = LinearRegressionGD(learning_rate=0.05, n_iterations=3000)
    model.fit(X_train_norm, y_train)

    return {
        "degree": degree,
        "model": model,
        "means": means,
        "scales": scales,
        "train_mse": mean_squared_error(y_train, model.predict(X_train_norm)),
        "test_mse": mean_squared_error(y_test, model.predict(X_test_norm)),
    }


results = [train_degree(degree) for degree in [1, 2, 10]]
for result in results:
    print(
        f"degree {result['degree']}: "
        f"train MSE={result['train_mse']:.3f}, "
        f"test MSE={result['test_mse']:.3f}"
    )

## Plot The Fits

The degree-1 model is usually too simple. Degree 2 matches the true pattern. Degree 10 has enough flexibility to chase noise.

In [ ]:
x_grid = np.linspace(float(np.min(X)), float(np.max(X)), 300).reshape(-1, 1)

fig, ax = plt.subplots()
ax.scatter(X_train[:, 0], y_train, label="train", alpha=0.7)
ax.scatter(X_test[:, 0], y_test, label="test", alpha=0.7)

for result in results:
    x_grid_poly = polynomial_features(x_grid, degree=result["degree"])
    x_grid_norm = transform_features(x_grid_poly, result["means"], result["scales"])
    y_grid = result["model"].predict(x_grid_norm)
    ax.plot(x_grid[:, 0], y_grid, label=f"degree {result['degree']}")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Polynomial degree comparison")
ax.legend()
plt.show()

## Takeaway

Higher degree is not automatically better. The model should fit the real pattern, not just the training noise. That is why test error matters.